In [8]:
# !pip install oracledb

import pandas as pd
import oracledb

df = pd.read_excel('./data/전처리완료_공고데이터.xlsx')

# 1. 컬럼명 변경
df = df.rename(columns={
    '기업명': 'COMPANY_NAME',
    '공고등록일': 'POSTED_DATE',
    '지원마감': 'DEADLINE',
    '지역': 'REGION',
    '고용형태': 'EMPLOYMENT_TYPE',
    '학력': 'EDUCATION',
    '직무': 'JOB_POSITION',
    '스킬': 'TECH_STACK',
    '급여': 'SALARY',
    '지원방법': 'POSTING_URL'
})

# 2. DB용 컬럼만 추리기
df_db = df[
    [
        'COMPANY_NAME',
        'POSTED_DATE',
        'DEADLINE',
        'REGION',
        'EMPLOYMENT_TYPE',
        'EDUCATION',
        'JOB_POSITION',
        'TECH_STACK',
        'SALARY',
        'POSTING_URL'
    ]
].copy()

# 3. 오라클 연결
conn = oracledb.connect(
    user='campus_24KDT_LI8_p2_1',
    password='smhrd1',
    dsn="project-db-campus.smhrd.com:1524/xe"
)
cursor = conn.cursor()

# 4. INSERT SQL
sql = """
INSERT INTO JOB_POSTING
(
    POSTING_ID,
    COMPANY_NAME,
    POSTED_DATE,
    DEADLINE,
    REGION,
    EMPLOYMENT_TYPE,
    EDUCATION,
    JOB_POSITION,
    TECH_STACK,
    SALARY,
    POSTING_URL
)
VALUES
(
    SEQ_JOB_POSTING.NEXTVAL,
    :1, :2, :3, :4, :5, :6, :7, :8, :9, :10
)
"""

# 5. 데이터 변환
data = []
for row in df_db.itertuples(index=False):
    data.append((
        row.COMPANY_NAME,
        row.POSTED_DATE.to_pydatetime() if pd.notna(row.POSTED_DATE) else None,
        row.DEADLINE.to_pydatetime() if pd.notna(row.DEADLINE) else None,
        row.REGION,
        row.EMPLOYMENT_TYPE,
        row.EDUCATION,
        row.JOB_POSITION,
        row.TECH_STACK,
        row.SALARY,
        row.POSTING_URL
    ))

# 6. DB insert
cursor.executemany(sql, data)
conn.commit()

# 7. 확인
cursor.execute("SELECT COUNT(*) FROM JOB_POSTING")
print(cursor.fetchone())

# 8. 종료
cursor.close()
conn.close()

(884,)


In [10]:
df_db.to_excel('DB연동_공고데이터.xlsx', index=False)

In [5]:
import oracledb

oracledb.init_oracle_client(
    lib_dir=r"C:\instantclient-basic-windows.x64-23.26.1.0.0\instantclient_23_0"
)